# Experiment No. 5: Sequential Data Modeling using Vanilla RNN

## Title
Sequential Time-Series Forecasting using Vanilla Recurrent Neural Networks (RNN).

## Aim


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Ensure project src/ directory is importable
module_path = os.path.abspath(os.path.join('..', 'src'))
if module_path not in sys.path:
    sys.path.append(module_path)

from data_loader import load_config, get_or_create_energy_data, explore_time_series_summary
from time_series_preprocessing import TimeSeriesScaler, clean_and_interpolate_series
from sequence_generator import create_sequences, split_time_series_sequences, prepare_pytorch_dataloaders
from model_builder import build_rnn_model, summarize_model
from training import train_model
from evaluation import evaluate_model_on_test_set, forecast_future_timesteps, generate_metrics_report
from visualization import plot_training_history, plot_predictions_vs_actual, plot_error_analysis, plot_autocorrelation, plot_future_forecast



## 1. Load Configuration & Data Acquisition

In [ ]:
config = load_config('../config/hyperparameters.json')
df = get_or_create_energy_data(config)
explore_time_series_summary(df, target_col='Energy_Consumption_kWh')


## 2. Time Series Preprocessing & MinMaxScaler Normalization

In [ ]:
raw_energy_series = clean_and_interpolate_series(df['Energy_Consumption_kWh'])
scaler = TimeSeriesScaler(feature_range=(0, 1))
scaled_energy = scaler.fit_transform(raw_energy_series.values)

plt.figure(figsize=(12, 4))
plt.plot(df['Datetime'], raw_energy_series, color='#1F77B4', label='Household Household Energy Consumption (kWh)')
plt.title('Household Energy Consumption Time Series (1,000 Days)', fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Energy Consumption (kWh)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)


## 3. Sliding Window Sequence Generation ($w=24$)

In [ ]:
lookback = 24
X_seq, y_seq = create_sequences(scaled_energy, lookback=lookback, lookahead=1)
X_tr, y_tr, X_val, y_val, X_te, y_te = split_time_series_sequences(X_seq, y_seq, train_ratio=0.8, val_ratio=0.1)
train_loader, val_loader, test_loader = prepare_pytorch_dataloaders(X_tr, y_tr, X_val, y_val, X_te, y_te, batch_size=32)



## 4. Construct & Train Stacked Vanilla RNN Model

In [ ]:
model = build_rnn_model(config)
summarize_model(model)



## 5. Model Evaluation & Visualizations

In [ ]:
metrics = evaluate_model_on_test_set(model, test_loader, scaler, config)

print(f"Test RMSE : {metrics['rmse']:.4f} kWh")
print(f"Test MAE  : {metrics['mae']:.4f} kWh")
print(f"Test MAPE : {metrics['mape']:.2f}%")
print(f"Test R²   : {metrics['r2']:.4f}")

plot_training_history(history, '../results/training_history.png')
plot_predictions_vs_actual(metrics['y_true'], metrics['y_pred'], '../results/predictions_vs_actual.png')
plot_error_analysis(metrics['y_true'], metrics['y_pred'], metrics['residuals'], '../results/error_analysis.png')
